In [3]:
"""
Bienestar subjetivo (SWB) y felicidad en trabajadores remunerados · Datos ESS

Contenido:
    1. SEM del SWB: comparación de dos modelos (A vs. B), ponderado y sin ponderar
    2. Regresión lineal ponderada (WLS) de la felicidad en trabajadores remunerados
       + logit ordinal como análisis de robustez
    3. Moderación por género y edad (interacciones)
    4. Comparación de los modelos de moderación

Requisitos (solo la primera vez):
    pip install semopy statsmodels pandas numpy scipy
"""

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import semopy
import statsmodels.api as sm
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel

warnings.filterwarnings("ignore")


# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================

# Ruta relativa al repositorio (evita exponer rutas personales al subirlo a GitHub)
RUTA_CSV = Path("C:/Users/Usuario/Downloads/Datos brutos ESS.csv")

# --- SEM del SWB --------------------------------------------------------------
VARS_SWB = ["happy", "stflife", "wrhpp", "enjlf",
            "fltsd", "fltdpr", "fltlnl", "cldgng", "anweight"]
VARS_A_INVERTIR = ["fltsd", "fltdpr", "fltlnl", "cldgng"]   # escala 1-4 -> 5 - x

# Códigos de no respuesta específicos de cada variable
MISSING_POR_VARIABLE = {
    "happy":    [77, 88, 99],
    "stflife":  [77, 88, 99],
    "wrhpp":    [7, 8, 9],
    "enjlf":    [7, 8, 9],
    "fltsd":    [7, 8, 9],
    "fltdpr":   [7, 8, 9],
    "fltlnl":   [7, 8, 9],
    "cldgng":   [7, 8, 9],
    "anweight": [],
}

COLS_SEM = ["wrhpp", "enjlf", "fltsd_inv", "fltdpr_inv", "fltlnl_inv",
            "cldgng_inv", "stflife", "happy"]

# Modelo A: happy es el indicador directo del SWB
DESC_A = """
AP =~ wrhpp + enjlf
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ stflife + cldgng_inv
SWB =~ AP + AN + SV
happy ~ SWB
"""

# Modelo B: happy entra en AP y stflife es el indicador directo del SWB
DESC_B = """
AP =~ wrhpp + enjlf + happy
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ cldgng_inv
SWB =~ AP + AN + SV
stflife ~ SWB
"""

# --- Regresión (trabajadores remunerados) ------------------------------------
VIS_CONTINUAS = ["hincfel_inv", "wkhtot", "wkdcorga"]
VIS_MODELO = ["hincfel_inv_c", "wkhtot_c", "wrkctra_limited",
              "wrkctra_nocontract", "wkdcorga_c", "jbspv_r"]

NOMBRES_VIS = {
    "const":              "Constante",
    "hincfel_inv_c":      "Comodidad con los ingresos (hincfel invertida, 1 = muy difícil a 4 = cómodo)",
    "wkhtot_c":           "Horas trabajadas (wkhtot)",
    "wrkctra_limited":    "Contrato limitado vs. indefinido",
    "wrkctra_nocontract": "Sin contrato vs. indefinido",
    "wkdcorga_c":         "Autonomía organizativa (wkdcorga, 0-10)",
    "jbspv_r":            "Supervisa a otros (jbspv, 1 = sí)",
}


# =============================================================================
# 1. UTILIDADES GENERALES
# =============================================================================

def imprimir_titulo(texto):
    print("\n" + "=" * 60)
    print(texto)
    print("=" * 60)


def cargar_datos(ruta=RUTA_CSV):
    """Lee el CSV del ESS y limpia los nombres de columnas."""
    df = pd.read_csv(ruta, sep=";", decimal=",")
    df.columns = df.columns.str.strip().str.replace('"', "")
    return df


def a_numerico(df, columnas, missing=None):
    """
    Convierte columnas a numérico. `missing` es un dict {variable: [códigos]}
    cuyos códigos de no respuesta se sustituyen por NaN.
    """
    df = df.copy()
    for col in columnas:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if missing and missing.get(col):
            df[col] = df[col].replace(missing[col], np.nan)
    return df


def correlacion_ponderada(x, y, w):
    """Correlación de Pearson ponderada."""
    w = w / w.sum()
    mx, my = np.sum(w * x), np.sum(w * y)
    cov = np.sum(w * (x - mx) * (y - my))
    sx = np.sqrt(np.sum(w * (x - mx) ** 2))
    sy = np.sqrt(np.sum(w * (y - my) ** 2))
    return cov / (sx * sy)


def corr_ponderada_matriz(X, w):
    """Matriz de correlaciones ponderada."""
    C = np.cov(X.values.T, aweights=np.asarray(w, float))
    d = np.sqrt(np.diag(C))
    return pd.DataFrame(C / np.outer(d, d), index=X.columns, columns=X.columns)


def n_efectivo(w):
    """N efectivo de Kish."""
    w = np.asarray(w, float)
    return w.sum() ** 2 / (w ** 2).sum()


# =============================================================================
# 2. SEM DEL SWB
# =============================================================================

def preparar_datos_swb(df_raw):
    """Limpia códigos de no respuesta, elimina filas incompletas e invierte ítems negativos."""
    df = a_numerico(df_raw[VARS_SWB], VARS_SWB, missing=MISSING_POR_VARIABLE)
    if "cntry" in df_raw.columns:
        df["cntry"] = df_raw["cntry"].values
    df = df.dropna(subset=VARS_SWB).reset_index(drop=True)
    for v in VARS_A_INVERTIR:
        df[f"{v}_inv"] = 5 - df[v]
    print(f"N total tras limpieza: {len(df)}")
    return df


def ejecutar_sem(df, desc, columnas, titulo, ponderado=True):
    """Ajusta un SEM a partir de la matriz de correlaciones (ponderada o no)."""
    X = df[columnas]
    if ponderado:
        S = corr_ponderada_matriz(X, df["anweight"])
        n = int(n_efectivo(df["anweight"]))
    else:
        S = X.corr()
        n = len(X)

    model = semopy.Model(desc)
    model.fit(cov=S, n_samples=n)

    imprimir_titulo(f"{titulo}  (N bruto = {len(X)} | N en los índices = {n})")
    resultados = model.inspect(std_est=True).copy()
    resultados["p-value"] = resultados["p-value"].apply(
        lambda x: f"{float(x):.3f}" if x not in ["-", "", None] and pd.notna(x) else "-")
    print(resultados.to_string(index=True))

    print("\n=== ÍNDICES DE AJUSTE ===")
    print(semopy.calc_stats(model).T
          .drop(index=["AIC", "BIC", "LogLik"], errors="ignore").round(3))
    return model


def comparar_modelos(modelos, titulo, n, columnas):
    """
    Compara el ajuste de varios SEM. AIC/BIC se calculan desde χ²
    (los que devuelve semopy no son válidos con matriz de covarianzas).
    """
    p = len(columnas)
    momentos = p * (p + 1) // 2

    filas = {}
    for nombre, m in modelos.items():
        s = semopy.calc_stats(m).iloc[0]
        chi2, gl = float(s["chi2"]), float(s["DoF"])
        q = momentos - gl                       # parámetros libres
        filas[nombre] = {"χ²": chi2, "gl": gl,
                         "CFI": s.get("CFI"), "TLI": s.get("TLI"),
                         "RMSEA": s.get("RMSEA"),
                         "AIC": chi2 + 2 * q,
                         "BIC": chi2 + q * np.log(n)}
    tabla = pd.DataFrame(filas).T.astype(float)

    imprimir_titulo(titulo)
    print(tabla.round(3).to_string())

    nombres = list(modelos)
    if len(nombres) == 2:
        a, b = tabla.loc[nombres[0]], tabla.loc[nombres[1]]
        dchi, ddf = b["χ²"] - a["χ²"], b["gl"] - a["gl"]
        if ddf > 0:
            print(f"\nContraste {nombres[1]} vs {nombres[0]}: "
                  f"Δχ² = {dchi:.2f}, Δgl = {ddf:.0f}, p = {stats.chi2.sf(dchi, ddf):.4g}")
    print(f"\nMenor AIC: {tabla['AIC'].idxmin()} | Menor BIC: {tabla['BIC'].idxmin()}")
    return tabla


def diagnostico_peso(df):
    """Efecto de la ponderación sobre la correlación happy-stflife, global y por país."""
    imprimir_titulo("DIAGNÓSTICO DEL PESO: happy vs stflife")
    r_sin = df["happy"].corr(df["stflife"])
    r_pon = correlacion_ponderada(df["happy"], df["stflife"], df["anweight"])
    print(f"r sin ponderar = {r_sin:.3f} | r ponderada = {r_pon:.3f}")
    print(f"N efectivo (Kish) = {n_efectivo(df['anweight']):.0f} de {len(df)}")
    print("Peso mín/mediana/máx =",
          round(df["anweight"].min(), 3),
          round(df["anweight"].median(), 3),
          round(df["anweight"].max(), 3))

    if "cntry" not in df.columns:
        return

    print("\nPaíses incluidos en el SEM:")
    print(df["cntry"].value_counts())

    res = [{"cntry": c, "n": len(g),
            "peso_medio": g["anweight"].mean(),
            "r_sin": g["happy"].corr(g["stflife"]),
            "r_pond": correlacion_ponderada(g["happy"], g["stflife"], g["anweight"])}
           for c, g in df.groupby("cntry") if len(g) > 100]
    print("\n", pd.DataFrame(res).sort_values("peso_medio", ascending=False)
                  .round(3).to_string(index=False))


def analisis_sem(df_raw):
    df_swb = preparar_datos_swb(df_raw)

    # Análisis principal: ponderado
    mA = ejecutar_sem(df_swb, DESC_A, COLS_SEM, "MODELO A: happy carga en SWB")
    mB = ejecutar_sem(df_swb, DESC_B, COLS_SEM, "MODELO B: stflife carga en SWB")
    comparar_modelos({"A (happy en SWB)": mA, "B (stflife en SWB)": mB},
                     "COMPARACIÓN PONDERADA",
                     int(n_efectivo(df_swb["anweight"])), COLS_SEM)

    # Sensibilidad: sin ponderar
    mA_s = ejecutar_sem(df_swb, DESC_A, COLS_SEM, "MODELO A sin ponderar", ponderado=False)
    mB_s = ejecutar_sem(df_swb, DESC_B, COLS_SEM, "MODELO B sin ponderar", ponderado=False)
    comparar_modelos({"A (happy en SWB)": mA_s, "B (stflife en SWB)": mB_s},
                     "COMPARACIÓN SIN PONDERAR", len(df_swb), COLS_SEM)

    diagnostico_peso(df_swb)


# =============================================================================
# 3. REGRESIÓN WLS (TRABAJADORES REMUNERADOS)
# =============================================================================

def preparar_trabajo_pagado(df_raw, extra_cols=()):
    """
    Filtra trabajadores remunerados (mnactic = 1), elimina códigos de no respuesta,
    recodifica y centra las VIs. `extra_cols` permite añadir moderadores.
    """
    cols = ["happy", "mnactic", "hincfel", "wkhtot", "wrkctra",
            "wkdcorga", "jbspv", "anweight", *extra_cols]
    df = a_numerico(df_raw[cols], cols)
    df["cntry"] = df_raw["cntry"]       # texto: se añade tras convertir a numérico

    # Muestra y códigos de no respuesta
    df = df[df["mnactic"] == 1]
    df = df[df["happy"].le(10)]
    df = df[df["hincfel"].isin([1, 2, 3, 4])]
    df = df[df["wkhtot"].between(1, 80)]
    df = df[df["wrkctra"].isin([1, 2, 3])]
    df = df[df["wkdcorga"].le(10)]
    df = df[df["jbspv"].isin([1, 2])]

    # Moderadores: quitar no respuesta (y la categoría 3 de género, solo 68 casos)
    if "agegroup" in df:
        df = df[df["agegroup"].between(1, 7)]
    if "nobingnd" in df:
        df = df[df["nobingnd"].isin([1, 2, 3])]

    df = df.dropna().reset_index(drop=True)

    # Recodificaciones
    df["hincfel_inv"] = 5 - df["hincfel"]                            # 4 = cómodo
    df["jbspv_r"] = (df["jbspv"] == 1).astype(int)                   # 1 = supervisa
    df["wrkctra_limited"] = (df["wrkctra"] == 2).astype(int)         # ref = indefinido
    df["wrkctra_nocontract"] = (df["wrkctra"] == 3).astype(int)

    # Centrado
    for var in VIS_CONTINUAS:
        df[f"{var}_c"] = df[var] - df[var].mean()
    return df


def ajustar_wls(df, X):
    """WLS con anweight y errores estándar agrupados por país (t con gl = clusters - 1)."""
    res = sm.WLS(df["happy"].astype(float), X.astype(float),
                 weights=df["anweight"].astype(float)).fit(
        cov_type="cluster",
        cov_kwds={"groups": pd.factorize(df["cntry"])[0]},
        use_t=True)
    res.n_paises = df["cntry"].nunique()
    return res


def tabla_coeficientes(modelo, nombres=None, ocultar="pais_"):
    """Tabla B, SE, t, p (sin las dummies de país)."""
    idx = [v for v in modelo.params.index if not v.startswith(ocultar)]
    nombres_var = [nombres.get(v, v) if nombres else v for v in idx]
    return pd.DataFrame({
        "Variable": nombres_var,
        "B":  modelo.params[idx].values.round(3),
        "SE": modelo.bse[idx].values.round(3),
        "t":  modelo.tvalues[idx].values.round(3),
        "p":  [f"{p:.3f}" for p in modelo.pvalues[idx]],
    })


def imprimir_resumen_modelo(modelo, nombres=None):
    print(tabla_coeficientes(modelo, nombres).to_string(index=False))
    print(f"\nN = {int(modelo.nobs)} | países (clusters) = {modelo.n_paises}")
    print(f"R² ponderado = {modelo.rsquared:.3f} | R² ajustado = {modelo.rsquared_adj:.3f}")


def descriptivos_trabajo_pagado(df):
    print(f"N grupo 1 (trabajo pagado) tras limpieza: {len(df)}")
    print("\nEstadísticos descriptivos (continuas):")
    print(df[["happy", "hincfel_inv", "wkhtot", "wkdcorga"]].describe().round(3))
    print("\nTipo de contrato (proporciones):")
    print(df["wrkctra"].value_counts(normalize=True).sort_index().round(3))
    print("\nSupervisa a otros (proporción):")
    print(df["jbspv_r"].value_counts(normalize=True).sort_index().round(3))


def analisis_regresion(df_raw):
    df = preparar_trabajo_pagado(df_raw)
    descriptivos_trabajo_pagado(df)

    # Modelo principal: WLS
    X = sm.add_constant(df[VIS_MODELO])
    modelo = ajustar_wls(df, X)
    imprimir_titulo("MODELO 2 – REGRESIÓN LINEAL PONDERADA (WLS)\n"
                    "Grupo 1 mnactic: trabajadores remunerados")
    imprimir_resumen_modelo(modelo, NOMBRES_VIS)
    print("\nNota: referencia wrkctra = contrato indefinido (1)")
    print("Nota: jbspv recodificada: 1 = supervisa, 0 = no supervisa")

    # Robustez: logit ordinal (sin pesos; EE agrupados por país)
    imprimir_titulo("ROBUSTEZ – LOGIT ORDINAL")
    ord_res = OrderedModel(
        df["happy"].astype(int), df[VIS_MODELO].astype(float), distr="logit"
    ).fit(method="bfgs", disp=False,
          cov_type="cluster", cov_kwds={"groups": pd.factorize(df["cntry"])[0]})
    print(ord_res.summary().tables[1])
    return modelo


# =============================================================================
# 4. MODERACIÓN POR GÉNERO Y EDAD
# =============================================================================

def ajustar_moderacion(df, moderadores):
    """
    moderadores: dict {prefijo_dummy: columna}, p. ej. {'genero': 'nobingnd'}.
    Devuelve el WLS con efectos principales, dummies e interacciones VI x moderador.
    """
    dummies = {
        pref: pd.get_dummies(df[col], prefix=pref, drop_first=True).astype(float)
        for pref, col in moderadores.items()
    }

    interacciones = [
        pd.DataFrame({f"{vi}_x_{c}": df[vi].values * dum[c].values
                      for vi in VIS_MODELO for c in dum.columns})
        for dum in dummies.values()
    ]

    X = pd.concat([df[VIS_MODELO], *dummies.values(), *interacciones], axis=1).astype(float)
    return ajustar_wls(df, sm.add_constant(X))


def test_interacciones(modelo, etiqueta):
    """Test de Wald conjunto del bloque de interacciones, global y por variable."""
    inter = [n for n in modelo.params.index if "_x_" in n]

    def wald(nombres):
        return modelo.wald_test(", ".join(f"{n} = 0" for n in nombres),
                                use_f=True, scalar=True)

    t = wald(inter)
    print(f"\n{etiqueta} | bloque completo: F({int(t.df_num)}, {int(t.df_denom)}) = "
          f"{float(t.statistic):.2f}, p = {float(t.pvalue):.4f}")

    for vi in VIS_MODELO:
        sub = [n for n in inter if n.startswith(f"{vi}_x_")]
        if sub:
            t = wald(sub)
            print(f"   {vi:<20} F({int(t.df_num)}, {int(t.df_denom)}) = "
                  f"{float(t.statistic):.2f}, p = {float(t.pvalue):.4f}")


def analisis_moderacion(df_raw):
    df = preparar_trabajo_pagado(df_raw, extra_cols=["nobingnd", "agegroup"])
    print(f"N grupo 1 (trabajo pagado) tras limpieza: {len(df)}")

    especificaciones = {
        "3A": ("MODELO 3A – MODERACIÓN POR GÉNERO (nobingnd)", "3A género",
               {"genero": "nobingnd"}),
        "3B": ("MODELO 3B – MODERACIÓN POR EDAD (agegroup)", "3B edad",
               {"edad": "agegroup"}),
        "3C": ("MODELO 3C – MODERACIÓN CONJUNTA GÉNERO Y EDAD", "3C conjunto",
               {"genero": "nobingnd", "edad": "agegroup"}),
    }

    modelos = {}
    for clave, (titulo, etiqueta, moderadores) in especificaciones.items():
        m = ajustar_moderacion(df, moderadores)
        imprimir_titulo(titulo)
        imprimir_resumen_modelo(m)
        test_interacciones(m, etiqueta)
        modelos[etiqueta.replace(" ", " (", 1) + ")"] = m   # "3A género" -> "3A (género)"

    # Comparación de modelos
    comparacion = pd.DataFrame({
        "Modelo":      list(modelos),
        "R²":          [m.rsquared for m in modelos.values()],
        "R² ajustado": [m.rsquared_adj for m in modelos.values()],
    })
    imprimir_titulo("COMPARACIÓN DE MODELOS")
    print(comparacion.round(3).to_string(index=False))

    n_paises = next(iter(modelos.values())).n_paises
    print("\nNotas:")
    print("- wrkctra: ref = contrato indefinido (1); limited = contrato temporal; nocontract = sin contrato")
    print("- jbspv: 1 = supervisa, 0 = no supervisa")
    print("- hincfel_inv: 1 = muy difícil, 4 = vive cómodamente (B positivo = más comodidad, más felicidad)")
    print("- wkdcorga: 0 = ninguna influencia, 10 = control total")
    print(f"- Errores estándar agrupados por país ({n_paises} clusters); "
          f"p-values con t({n_paises - 1})")
    return modelos


# =============================================================================
# 5. EJECUCIÓN
# =============================================================================

def main():
    df_raw = cargar_datos()
    analisis_sem(df_raw)
    analisis_regresion(df_raw)
    analisis_moderacion(df_raw)


if __name__ == "__main__":
    main()

N total tras limpieza: 34701

MODELO A: happy carga en SWB  (N bruto = 34701 | N en los índices = 22608)
          lval  op        rval      Estimate      Est. Std  Std. Err     z-value p-value
0           AP   ~         SWB  1.000000e+00  7.419976e-01         -           -       -
1           AN   ~         SWB  9.000025e-01  7.240393e-01  0.012908   69.723806   0.000
2           SV   ~         SWB  1.256310e+00  1.000000e+00  0.014666   85.658615   0.000
3        wrhpp   ~          AP  1.000000e+00  8.250185e-01         -           -       -
4        enjlf   ~          AP  9.630105e-01  7.946098e-01  0.009768   98.592498   0.000
5    fltsd_inv   ~          AN  1.000000e+00  7.609780e-01         -           -       -
6   fltdpr_inv   ~          AN  9.362814e-01  7.124915e-01  0.010448     89.6168   0.000
7   fltlnl_inv   ~          AN  8.582825e-01  6.531580e-01  0.010196   84.180479   0.000
8      stflife   ~          SV  1.000000e+00  7.692440e-01         -           -       -
9   c